In [60]:
import pandas as pd
from pathlib import Path
import sqlite3

datasets_base_path = Path('datasets')
datasets_base_path.mkdir(exist_ok=True)

con = sqlite3.connect('omop.db')
cur = con.cursor()

In [3]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cur.fetchall()
print(tables)

[('concept',), ('concept_ancestor',), ('concept_class',), ('concept_relationship',), ('concept_synonym',), ('domain',), ('drug_strength',), ('relationship',), ('source_to_concept_map',), ('vocabulary',)]


In [32]:
# Collect concept_id + concept_name from standard vocabularies
sql_prompt = f"""
SELECT 
    c.concept_id, 
    c.concept_name
FROM concept c
WHERE standard_concept = 'S'
"""
standard_concept_df = pd.read_sql(sql_prompt, con)

# Join with synonyms
sql_prompt = f"""
SELECT 
    c1.concept_id AS concept_id, 
    c2.concept_synonym_name AS concept_name
FROM concept c1
INNER JOIN concept_synonym c2 ON c1.concept_id = c2.concept_id
"""
standard_concept_synonyms_df = pd.read_sql(sql_prompt, con)

In [57]:
standard_and_synonyms_df = pd.concat([standard_concept_df, standard_concept_synonyms_df])
standard_and_synonyms_df = standard_and_synonyms_df.dropna()
standard_and_synonyms_df.reset_index(drop=True, inplace=True)

In [61]:
standard_and_synonyms_df.to_csv(datasets_base_path / 'train.csv', index=False)

In [50]:
# Collect concepts from other vocabularies that are mapped to standard vocabularies and create an evaluation dataset
sql_prompt = f"""
SELECT
    cr.concept_id_2 AS concept_id,
    c1.concept_name AS concept_name
FROM concept_relationship cr
LEFT JOIN concept c1 ON cr.concept_id_1 = c1.concept_id
LEFT JOIN concept c2 ON cr.concept_id_2 = c2.concept_id
WHERE cr.relationship_id = 'Maps to'
    AND cr.concept_id_1 != cr.concept_id_2
    AND c1.concept_name != c2.concept_name
    AND c2.standard_concept = 'S'
"""

concept_relationship_df = pd.read_sql(sql_prompt, con)
concept_relationship_df

,concept_id,concept_name
0,45769156,Liddle syndrome
1,4022675,Clostridium difficile toxin A + B
2,4022675,Bordetella pertussis filamentous hemagglutinin...
3,4022675,Coxsackievirus + Echovirus IgA
4,4022675,Collagen + procollagen
...,...,...
213070,19024535,Carbazochrome-containing product
213071,19024535,Carbazochrome sodium sulfonate product
213072,37312441,Back performance-feedback physical therapy sys...
213073,19068969,Hexoprenaline-containing product


In [51]:
concept_relationship_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213075 entries, 0 to 213074
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   concept_id    213075 non-null  object
 1   concept_name  213075 non-null  object
dtypes: object(2)
memory usage: 3.3+ MB


In [62]:
concept_relationship_df.to_csv(datasets_base_path / 'eval.csv', index=False)

In [66]:
standard_and_synonyms_df['concept_id']

0          42538812
1           4084170
2           4085530
3           4085038
4           4085531
             ...   
3150633     1259883
3150634     3048022
3150635     3048022
3150636     3048022
3150637     3048022
Name: concept_id, Length: 3150638, dtype: object